# Prompting Patterns

Explore how system prompts, few-shot examples, structured prompt layouts, and structured outputs shape agent behaviour.

**Key learnings**
- System prompts set role and style before the user message
- Few-shot examples steer format without fine-tuning
- Pydantic `response_format` returns typed structured data


In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Basic prompting

Without a system prompt the model answers in a generic style. Adding a system prompt changes voice and creativity.


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model="gpt-5-nano")

question = HumanMessage(content="What's the capital of the moon?")

# Default agent: no system prompt — generic assistant behaviour
response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)


In [ ]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Few-shot examples

Embedding example Q&A pairs in the system prompt teaches the model the expected answer style (here: invented city names).


In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Structured prompts

Ask for a fixed field layout in the prompt so free-text answers stay consistent and easy to parse by eye.


In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Structured output

Pass a Pydantic model as `response_format` so the agent returns a typed object under `response["structured_response"]` instead of free text.


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

# Schema the model must fill — becomes response["structured_response"]
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response["structured_response"])


In [ ]:
response["structured_response"].name

In [ ]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")